# Experiment 3 — Hist3 arm (Qwen2.5-Omni-3B, IEMOCAP)

**Single new arm only.** NoHist (Control) and Hist8 (Treatment) are frozen — their complete
Experiment‑2 artifacts already exist locally (`kaggle_prep/experiment2_results/`) and are
**not** recomputed here.

| | NoHist (frozen) | Hist8 (frozen) | **Hist3 (this run)** |
|---|---|---|---|
| Target audio/transcript/gold | test.json | identical | identical |
| History window | none | 8 preceding turns | **3 preceding turns** |
| Current turn in history | — | excluded | excluded |
| Speaker identity | — | anonymized `Speaker_1`/`Speaker_2`, persistent per dialogue | identical scheme |
| Model / decoding / seed | Qwen2.5-Omni-3B, greedy, seed=11 | identical | identical |
| Parser / scoring | `iemocap_eval_lib.py` (sha256 `24fac1f49d6f049c...`) | identical file | **identical file, unmodified** |

Validated locally (10/10 checks, `kaggle_prep/validate_history3.py`) before this notebook was
built: same 1,622 ids in the same order as NoHist/Hist8; every field except `history_context`
byte-identical across all three; Hist3's window is exactly the most-recent 3 turns of Hist8's
window for every target (reconstructed independently from the raw dialogue CSV, not by string
parsing); the target/current turn never appears in its own window; dialogue-opener set (22
targets, 0 preceding turns) is identical between Hist3/Hist8; speaker anonymization
byte-for-byte reproduced by a fresh, independent recomputation; behavioral leakage test
(scrambling/dropping gold emotion + VAD columns) confirms the window-construction output is
provably unaffected by them for window=3 specifically.

**This notebook prints only Hist3's own metrics.** It does NOT compute or claim any
NoHist-vs-Hist3 or Hist8-vs-Hist3 delta — that three-way paired comparison is done locally,
after this run's artifacts are downloaded, exactly as was done for the Experiment‑2 diagnostic.


## 0 · Kaggle setup

1. **Add data:** a dataset containing `test.json`, `test_tiny.json`, `test_history3.json`,
   `test_tiny_history3.json`, `manifest.json`, `manifest_history3.json`, and `audio/`
   (same 1,622 Session‑5 wavs as before — reuse the existing dataset, just add the two new
   `*history3.json` files to it; no re-upload of `test_history8.json` or audio needed).
2. **Accelerator:** GPU **T4 x2**. **Internet:** On. No secret required.
3. Run cells top to bottom. Smoke test (36 ids) first; review before the full run
   (1,622 ids, single arm — expect roughly half the wall-clock time of Experiment 2's
   two-arm run, since only one arm runs here).

## ⚠️ Output persistence (same discipline as Experiment 2 — do not skip)

- Every 200 rows, and at completion, the run refreshes ONE archive,
  `/kaggle/working/results_bundle.zip`. Download it from **Data → Output** at least once
  mid-run as a live backup.
- Before closing the session: **Save Version → "Save & Run All (Commit)"** (not Quick Save).
  Verify the committed Version's Output tab lists `results/pred_hist3_newparser.jsonl` and
  `results/predictions_hist3_newparser.json` — the last cell checks this for you and prints
  an explicit checklist.


In [15]:
# ============================ CONFIG ============================
MODEL_ID       = "Qwen/Qwen2.5-Omni-3B"   # fixed -- identical weights to NoHist/Hist8
WINDOW         = 3                         # documentation only; already baked into test_history3.json
ARM_TAG        = "hist3_newparser"

RUN_SMOKE_TEST = True
RUN_FULL       = True
FORCE_FULL     = False

LIMIT          = None
MAX_NEW_TOKENS = 12       # identical to NoHist/Hist8
FORCE_FP16     = None
SEED           = 11       # identical to NoHist/Hist8

OUTPUT_DIR     = "/kaggle/working/results"

EVAL_LIB_SHA256_EXPECTED = "24fac1f49d6f049c1485b4769ce523a292ae692fad4657a9fab7be6af6034298"  # == the sha256 recorded for Hist8 in Experiment 2
print("MODEL_ID:", MODEL_ID, "| WINDOW:", WINDOW, "| SEED:", SEED, "| ARM_TAG:", ARM_TAG)


MODEL_ID: Qwen/Qwen2.5-Omni-3B | WINDOW: 3 | SEED: 11 | ARM_TAG: hist3_newparser


In [16]:
import subprocess, sys
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)
pip("-U", "transformers>=4.52.3", "accelerate>=0.34", "qwen-omni-utils", "librosa", "soundfile")
# Pre-applied fix (needed in Experiment 2's Kaggle image): librosa can pull a scipy upgrade
# that becomes ABI-inconsistent with the preinstalled numpy, raising
# "numpy._core._multiarray_umath has no attribute '_blas_supports_fpe'" at import time.
pip("-U", "--force-reinstall", "numpy", "scipy")
print("pip done")


pip done


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.67.0 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.3 which is incompatible.
ydata-profiling 4.18.4 requires scipy<1.17,>=1.8, but you have scipy 1.18.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60

In [17]:
# ================= output-persistence safety net =================
import os, json, time, random, hashlib, base64
import numpy as np
import torch

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

import transformers
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("CUDA        :", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"))
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print("compute cap :", cap)
    _is_t4 = cap[0] == 7
else:
    _is_t4 = False
USE_FP16 = _is_t4 if FORCE_FP16 is None else FORCE_FP16
DTYPE = torch.float16 if USE_FP16 else torch.bfloat16
print("dtype       :", DTYPE)
os.makedirs(OUTPUT_DIR, exist_ok=True)

import zipfile
ZIP_PATH = "/kaggle/working/results_bundle.zip"

def make_zip_bundle(note=""):
    tmp = ZIP_PATH + ".tmp"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(OUTPUT_DIR):
            for fn in files:
                fp = os.path.join(root, fn)
                zf.write(fp, os.path.relpath(fp, "/kaggle/working"))
    os.replace(tmp, ZIP_PATH)
    sz = os.path.getsize(ZIP_PATH)
    print(f"[checkpoint zip] {ZIP_PATH} ({sz/1e6:.2f} MB){' -- ' + note if note else ''} "
          f"-- download it now from Data > Output if you want an immediate local copy")
    return ZIP_PATH

make_zip_bundle("initial (empty) checkpoint")


torch       : 2.10.0+cu128
transformers: 5.17.0
CUDA        : True | GPUs: 2 | Tesla T4
compute cap : (7, 5)
dtype       : torch.float16
[checkpoint zip] /kaggle/working/results_bundle.zip (0.05 MB) -- initial (empty) checkpoint -- download it now from Data > Output if you want an immediate local copy


'/kaggle/working/results_bundle.zip'

In [18]:
# ---- locate the uploaded dataset (needs Control + Hist3 files + audio/) ----
CANDIDATE_ROOTS = []
for base in ["/kaggle/input"]:
    if os.path.isdir(base):
        for d in sorted(os.listdir(base)):
            CANDIDATE_ROOTS.append(os.path.join(base, d))

REQUIRED_FILES = ["test.json", "test_history3.json", "test_tiny.json", "test_tiny_history3.json"]

def _find_input_dir():
    for root in CANDIDATE_ROOTS:
        for dirpath, _, files in os.walk(root):
            if all(f in files for f in REQUIRED_FILES) and os.path.isdir(os.path.join(dirpath, "audio")):
                return dirpath
    raise FileNotFoundError(
        f"Could not find all of {REQUIRED_FILES} + audio/ under one directory in /kaggle/input. "
        f"Add a dataset containing the Experiment-3 files (built by kaggle_prep/build_history3_field.py) "
        f"alongside the existing audio/. Seen roots: {CANDIDATE_ROOTS}")

INPUT_DIR = _find_input_dir()
AUDIO_DIR = os.path.join(INPUT_DIR, "audio")
print("INPUT_DIR:", INPUT_DIR)
print("has audio/:", os.path.isdir(AUDIO_DIR), "| n wav:", len(os.listdir(AUDIO_DIR)))


INPUT_DIR: /kaggle/input/datasets/pranavjaiganesh/kagglev3/kaggle_upload
has audio/: True | n wav: 1622


In [19]:
# ---- HF auth (optional; Qwen is ungated) ----
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception as e:
    print("No HF_TOKEN secret (fine for Qwen):", e)


HF_TOKEN loaded from Kaggle Secrets.


In [20]:
# ================= iemocap_eval_lib.py, embedded verbatim as base64 =================
# UNCHANGED from Experiment 2 -- same file, same sha256, same parser/prompt/scoring code.
EVAL_LIB_B64 = "..."

_lib_bytes = base64.b64decode(EVAL_LIB_B64)
_lib_sha256 = hashlib.sha256(_lib_bytes).hexdigest()
print("iemocap_eval_lib.py decoded:", len(_lib_bytes), "bytes, sha256=", _lib_sha256)
assert _lib_sha256 == EVAL_LIB_SHA256_EXPECTED, (
    "iemocap_eval_lib.py content does not match Experiment 2's! "
    f"expected {EVAL_LIB_SHA256_EXPECTED}, got {_lib_sha256}")
print("sha256 matches Experiment 2's eval lib exactly -- same parser/scoring pipeline confirmed -- OK")

with open("/kaggle/working/iemocap_eval_lib.py", "wb") as f:
    f.write(_lib_bytes)
import importlib, sys
sys.path.insert(0, "/kaggle/working")
import iemocap_eval_lib as EVAL
importlib.reload(EVAL)
print("eval lib ready. labels:", EVAL.IEMOCAP_LABELS)


iemocap_eval_lib.py decoded: 24510 bytes, sha256= 24fac1f49d6f049c1485b4769ce523a292ae692fad4657a9fab7be6af6034298
sha256 matches Experiment 2's eval lib exactly -- same parser/scoring pipeline confirmed -- OK
eval lib ready. labels: ['happy', 'sad', 'neutral', 'angry', 'excited', 'frustrated']


In [21]:
# ---- load records, and ASSERT against Control at runtime (not just locally beforehand) ----
def load_records(name):
    with open(os.path.join(INPUT_DIR, name), encoding="utf-8") as f:
        return json.load(f)

CONTROL_TINY = load_records("test_tiny.json")
HIST3_TINY   = load_records("test_tiny_history3.json")
CONTROL_FULL = load_records("test.json")
HIST3_FULL   = load_records("test_history3.json")
if LIMIT:
    CONTROL_FULL = CONTROL_FULL[:LIMIT]
    HIST3_FULL = HIST3_FULL[:LIMIT]

def assert_paired(control, treatment, tag):
    assert len(control) == len(treatment), f"{tag}: length mismatch {len(control)} vs {len(treatment)}"
    cids = [r["id"] for r in control]; tids = [r["id"] for r in treatment]
    assert cids == tids, f"{tag}: id order mismatch"
    for c, t in zip(control, treatment):
        assert c["utterance"] == t["utterance"], f"{tag}: utterance diverged for {c['id']}"
        assert c["output"] == t["output"], f"{tag}: gold label diverged for {c['id']}"
        assert set(t.keys()) - set(c.keys()) == {"history_context"}, \
            f"{tag}: unexpected extra fields for {c['id']}: {set(t.keys())-set(c.keys())}"
    return cids

tiny_ids = assert_paired(CONTROL_TINY, HIST3_TINY, "tiny")
full_ids = assert_paired(CONTROL_FULL, HIST3_FULL, "full")
print(f"ASSERTIONS PASSED: tiny {len(tiny_ids)} ids paired & identical (except history_context); "
      f"full {len(full_ids)} ids paired & identical (except history_context)")
n_empty_hist = sum(1 for r in HIST3_FULL if r["history_context"] == "")
print(f"true dialogue-openers in full set (empty history_context): {n_empty_hist} (expect 22, same as Hist8)")


ASSERTIONS PASSED: tiny 36 ids paired & identical (except history_context); full 1622 ids paired & identical (except history_context)
true dialogue-openers in full set (empty history_context): 22 (expect 22, same as Hist8)


In [22]:
# ================= model loading (single arm) =================
_MODEL = {}

def load_model():
    from transformers import Qwen2_5OmniProcessor
    from qwen_omni_utils import process_mm_info
    import transformers as _tf
    OmniCls = getattr(_tf, "Qwen2_5OmniForConditionalGeneration",
                      getattr(_tf, "Qwen2_5OmniModel", None))
    if OmniCls is None:
        raise ImportError("This transformers build has no Qwen2.5-Omni class.")
    proc = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID)
    # NOTE: loading the full model (talker included), NOT passing enable_audio_output=False
    # and NOT calling disable_talker(): in Experiment 2's Kaggle image, generate() unconditionally
    # read self.talker.codec_pad_token before checking return_audio, so a talker-less model
    # crashed generate() even with return_audio=False. Loading it fully does not change the
    # generated TEXT (only self.thinker.generate() produces the returned tokens) -- same weights,
    # same output, just more VRAM used for an unused submodule. This is the fix already applied
    # (and proven to work) in Experiment 2's actual Kaggle run.
    model = OmniCls.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map="auto")
    model.eval()
    _MODEL.update(model=model, proc=proc, process_mm_info=process_mm_info)
    print("model loaded:", MODEL_ID, "(talker included, per Experiment-2 fix; identical weights)")

load_model()


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2543 [00:00<?, ?it/s]

[transformers] Qwen2_5OmniForConditionalGeneration LOAD REPORT from: Qwen/Qwen2.5-Omni-3B
Key                                                | Status     |  | 
---------------------------------------------------+------------+--+-
token2wav.code2wav_dit_model.rotary_embed.inv_freq | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model loaded: Qwen/Qwen2.5-Omni-3B (talker included, per Experiment-2 fix; identical weights)


In [23]:
# ================= per-utterance inference =================
QWEN_SYS = ("You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, "
            "capable of perceiving auditory and visual inputs, as well as generating text and speech.")

@torch.no_grad()
def infer_one(rec, history_context):
    prompt = EVAL.build_prompt(rec["utterance"], history_context)
    wav_path = os.path.join(AUDIO_DIR, rec["id"] + ".wav")
    model, proc, process_mm_info = _MODEL["model"], _MODEL["proc"], _MODEL["process_mm_info"]

    conv = [
        {"role": "system", "content": [{"type": "text", "text": QWEN_SYS}]},
        {"role": "user", "content": [
            {"type": "audio", "audio": wav_path},
            {"type": "text", "text": prompt}]},
    ]
    text = proc.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
    try:
        audios, images, videos = process_mm_info(conv, use_audio_in_video=False)
    except TypeError:
        audios, images, videos = process_mm_info(conv)
    try:
        inputs = proc(text=text, audio=audios, images=images, videos=videos,
                      return_tensors="pt", padding=True)
    except TypeError:
        inputs = proc(text=text, audios=audios, images=images, videos=videos,
                      return_tensors="pt", padding=True)
    inputs = inputs.to(model.device)
    try:
        inputs = inputs.to(model.dtype)
    except Exception:
        pass
    gkw = dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    try:
        out = model.generate(**inputs, return_audio=False, **gkw)
    except TypeError:
        out = model.generate(**inputs, **gkw)
    if isinstance(out, (tuple, list)):
        out = out[0]
    gen = out[:, inputs["input_ids"].shape[1]:]
    return proc.batch_decode(gen, skip_special_tokens=True,
                             clean_up_tokenization_spaces=False)[0].strip()

t0 = time.time()
_demo = infer_one(HIST3_TINY[0], HIST3_TINY[0]["history_context"])
print(f"1st inference OK in {time.time()-t0:.1f}s -> id={HIST3_TINY[0]['id']} "
      f"gold={HIST3_TINY[0]['output']} raw={_demo!r}")


1st inference OK in 1.1s -> id=Ses05F_impro01_F000 gold=neutral raw='Neutral. What do you think about this?'


In [24]:
# ================= SMOKE TEST (36 ids) =================
from tqdm.auto import tqdm

def run_set(records, desc):
    raw = []
    for r in tqdm(records, desc=desc):
        try:
            raw.append(infer_one(r, r["history_context"]))
        except Exception as e:
            print("infer error", r["id"], repr(e)); raw.append("")
    return raw

SMOKE_OK = None
if RUN_SMOKE_TEST:
    smoke_raw = run_set(HIST3_TINY, "smoke-hist3")
    m = EVAL.score_predictions_normalized(HIST3_TINY, smoke_raw)
    print(f"{'id':<24}{'gold':<12}{'pred':<12}{'tier':<22}{'raw'}")
    for i, uid in enumerate(tiny_ids):
        p = m["predictions"][i]
        print(f"{uid:<24}{p['gold']:<12}{p['normalized_prediction']:<12}{p['match_tier']:<22}{p['raw_generation']!r}")
    n_empty = sum(1 for x in smoke_raw if not x.strip())
    SMOKE_OK = n_empty <= 2
    print(f"\nHist3 smoke: WA={m['accuracy_WA']:.2f} macro-F1={m['macro_f1']:.2f} tiers={m['match_tier_counts']}")
    print(f"SMOKE_OK = {SMOKE_OK}")
    with open(os.path.join(OUTPUT_DIR, "smoke_hist3.jsonl"), "w", encoding="utf-8") as f:
        for uid, raw in zip(tiny_ids, smoke_raw): f.write(json.dumps({"id": uid, "raw": raw}, ensure_ascii=False)+"\n")
    make_zip_bundle("after smoke test")
else:
    print("smoke test skipped")


smoke-hist3:   0%|          | 0/36 [00:00<?, ?it/s]

id                      gold        pred        tier                  raw
Ses05F_impro01_F000     neutral     neutral     word_match            'Neutral. What do you think about this?'
Ses05F_impro01_M000     neutral     neutral     exact                 'neutral.'
Ses05F_impro01_F001     frustrated  frustrated  word_match            'Frustrated. If you have any other conversations or need further analysis, feel free to let me know.'
Ses05F_impro01_F002     frustrated  frustrated  word_match            'Frustrated. If you have any other conversations or need further analysis, feel free to share.'
Ses05F_impro01_F003     frustrated  frustrated  word_match            'Frustrated. If you have any other conversations or need further analysis, feel free to share.'
Ses05F_impro01_M003     neutral     frustrated  word_match            'Frustrated. If you have any other conversations or need further analysis, feel free to let me know.'
Ses05F_impro01_F004     frustrated  frustrated  word_match

In [25]:
# ================= FULL RUN (1622 ids, single arm) =================
if RUN_FULL and RUN_SMOKE_TEST and (SMOKE_OK is False) and not FORCE_FULL:
    raise RuntimeError("Smoke test looked broken. Not starting the full run. Set FORCE_FULL=True to override.")

PRED_JSONL = os.path.join(OUTPUT_DIR, f"pred_{ARM_TAG}.jsonl")
ZIP_EVERY = 200

def load_done(path):
    done = {}
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    o = json.loads(line); done[o["id"]] = o["raw"]
    return done

if RUN_FULL:
    done = load_done(PRED_JSONL)
    print(f"resuming: {len(done)} / {len(HIST3_FULL)} already done")
    t0 = time.time()
    with open(PRED_JSONL, "a", encoding="utf-8") as f:
        for i, r in enumerate(tqdm(HIST3_FULL, desc="hist3")):
            if r["id"] in done:
                continue
            try:
                raw = infer_one(r, r["history_context"])
            except Exception as e:
                print("infer error", r["id"], repr(e)); raw = ""
            done[r["id"]] = raw
            f.write(json.dumps({"id": r["id"], "raw": raw}, ensure_ascii=False) + "\n")
            f.flush()
            if (i + 1) % 100 == 0:
                el = time.time() - t0
                print(f"  {i+1}/{len(HIST3_FULL)}  elapsed {el/60:.1f}m  eta {el/(i+1)*(len(HIST3_FULL)-i-1)/60:.1f}m")
            if (i + 1) % ZIP_EVERY == 0:
                make_zip_bundle(f"mid-run checkpoint, {i+1}/{len(HIST3_FULL)}")
    print(f"done in {(time.time()-t0)/60:.1f} min -> {PRED_JSONL}")
    make_zip_bundle("full run complete")
else:
    print("full run skipped")


resuming: 0 / 1622 already done


hist3:   0%|          | 0/1622 [00:00<?, ?it/s]

  700/1622  elapsed 15.3m  eta 20.2m
  800/1622  elapsed 17.4m  eta 17.9m
[checkpoint zip] /kaggle/working/results_bundle.zip (0.05 MB) -- mid-run checkpoint, 800/1622 -- download it now from Data > Output if you want an immediate local copy
  900/1622  elapsed 20.0m  eta 16.0m
  1000/1622  elapsed 22.0m  eta 13.7m
[checkpoint zip] /kaggle/working/results_bundle.zip (0.06 MB) -- mid-run checkpoint, 1000/1622 -- download it now from Data > Output if you want an immediate local copy
  1100/1622  elapsed 24.3m  eta 11.5m
  1200/1622  elapsed 26.1m  eta 9.2m
[checkpoint zip] /kaggle/working/results_bundle.zip (0.06 MB) -- mid-run checkpoint, 1200/1622 -- download it now from Data > Output if you want an immediate local copy
  1300/1622  elapsed 28.6m  eta 7.1m
  1400/1622  elapsed 31.0m  eta 4.9m
[checkpoint zip] /kaggle/working/results_bundle.zip (0.06 MB) -- mid-run checkpoint, 1400/1622 -- download it now from Data > Output if you want an immediate local copy
  1500/1622  elapsed 33.2m 

In [26]:
# ================= SCORE + SAVE (Hist3's own metrics ONLY -- no comparison claims here) =================
import pandas as pd

def read_jsonl(path):
    d = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                o = json.loads(line); d[o["id"]] = o["raw"]
    return d

done = read_jsonl(PRED_JSONL)
missing = [r["id"] for r in HIST3_FULL if r["id"] not in done]
if missing:
    print(f"WARNING: {len(missing)} rows missing -- re-run the full-run cell to finish.")

scored = [r for r in HIST3_FULL if r["id"] in done]
raws = [done[r["id"]] for r in scored]
M = EVAL.score_predictions_normalized(scored, raws)

print("="*78)
print(f"HIST3 (window=3) -- N={M['n_samples']}" + (f"  (+{len(missing)} missing)" if missing else "  (full 1622)"))
print("="*78)
print(f"Accuracy/WA={M['accuracy_WA']:.2f}  UA={M['UA_unweighted_recall']:.2f}  "
      f"macro-F1={M['macro_f1']:.2f}  weighted-F1={M['weighted_f1']:.2f}")
print(f"match tiers: {M['match_tier_counts']}  ambiguous_multi_label={M['n_ambiguous_multi_label']}")
print(f"repo report_score: {M['repo_report_score']}")
print("\nThis cell reports ONLY Hist3's own numbers. NoHist-vs-Hist3 and Hist8-vs-Hist3 paired")
print("comparisons are computed LOCALLY afterward from the downloaded artifacts, against the")
print("already-saved Experiment-2 predictions -- not here, and not by re-running NoHist/Hist8.")
print("="*78)

with open(os.path.join(OUTPUT_DIR, f"metrics_{ARM_TAG}.json"), "w", encoding="utf-8") as f:
    json.dump({k: v for k, v in M.items() if k != "predictions"}, f, ensure_ascii=False, indent=2)
with open(os.path.join(OUTPUT_DIR, f"predictions_{ARM_TAG}.json"), "w", encoding="utf-8") as f:
    json.dump(M["predictions"], f, ensure_ascii=False, indent=2)
pd.DataFrame(M["confusion_matrix"]["rows_gold_cols_pred"],
             index=EVAL.IEMOCAP_LABELS, columns=EVAL.IEMOCAP_LABELS
             ).to_csv(os.path.join(OUTPUT_DIR, f"confusion_{ARM_TAG}.csv"))
run_meta = {"model_id": MODEL_ID, "seed": SEED, "max_new_tokens": MAX_NEW_TOKENS, "window": WINDOW,
            "arm_tag": ARM_TAG, "dtype": str(DTYPE), "transformers": transformers.__version__,
            "torch": torch.__version__, "eval_lib_sha256": _lib_sha256,
            "n_scored": len(scored), "missing": missing}
with open(os.path.join(OUTPUT_DIR, "run_meta_experiment3.json"), "w", encoding="utf-8") as f:
    json.dump(run_meta, f, ensure_ascii=False, indent=2)
print("\nsaved to", OUTPUT_DIR, "->", sorted(os.listdir(OUTPUT_DIR)))
make_zip_bundle("final, after scoring")


HIST3 (window=3) -- N=1622  (full 1622)
Accuracy/WA=44.45  UA=44.26  macro-F1=42.03  weighted-F1=41.72
match tiers: {'exact': 513, 'word_match': 1105, 'edit_distance_fallback': 4}  ambiguous_multi_label=0
repo report_score: {'Acc_SA': 44.451, 'F1_SA': 41.715, 'mode': 'test'}

This cell reports ONLY Hist3's own numbers. NoHist-vs-Hist3 and Hist8-vs-Hist3 paired
comparisons are computed LOCALLY afterward from the downloaded artifacts, against the
already-saved Experiment-2 predictions -- not here, and not by re-running NoHist/Hist8.

saved to /kaggle/working/results -> ['confusion_control_nohist_newparser.csv', 'confusion_hist3_newparser.csv', 'confusion_treatment_hist8_newparser.csv', 'metrics_control_nohist_newparser.json', 'metrics_hist3_newparser.json', 'metrics_treatment_hist8_newparser.json', 'paired_comparison.json', 'pred_control_nohist_newparser.jsonl', 'pred_hist3_newparser.jsonl', 'pred_treatment_hist8_newparser.jsonl', 'predictions_control_nohist_newparser.json', 'predictions

'/kaggle/working/results_bundle.zip'

In [27]:
# ================= FINAL PERSISTENCE CHECK -- do not skip this cell =================
def _line_count(path):
    if not os.path.exists(path):
        return None
    with open(path, encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

CRITICAL_FILES = {
    "Hist3 raw predictions (jsonl)": PRED_JSONL,
    "Hist3 per-row predictions (json, for local paired comparison)":
        os.path.join(OUTPUT_DIR, f"predictions_{ARM_TAG}.json"),
}
EXPECTED_ROWS = len(HIST3_FULL)

print("="*78)
print("FINAL PERSISTENCE CHECK")
print("="*78)
all_ok = True
for label, path in CRITICAL_FILES.items():
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    if not exists:
        print(f"[MISSING] {label}: {path}"); all_ok = False
    elif size == 0:
        print(f"[EMPTY]   {label}: {path}"); all_ok = False
    else:
        if path.endswith(".jsonl"):
            n = _line_count(path)
            note = f"{n}/{EXPECTED_ROWS} rows" + ("" if n == EXPECTED_ROWS else "  <-- INCOMPLETE")
            if n != EXPECTED_ROWS: all_ok = False
            print(f"[ok]      {label}: {size/1e6:.2f} MB, {note}")
        else:
            print(f"[ok]      {label}: {size/1e3:.1f} KB")

zpath = make_zip_bundle("FINAL")
print(f"\nfull results/ listing: {sorted(os.listdir(OUTPUT_DIR))}")

print("\n" + "="*78)
print("ALL CRITICAL FILES PRESENT AND COMPLETE." if all_ok else
      "CRITICAL: one or more files above are MISSING/EMPTY/INCOMPLETE. Do not close this "
      "session yet -- re-run the full-run cell to finish, then re-run this cell.")
print("="*78)
print("""
NOW DO ONE OR BOTH OF THE FOLLOWING BEFORE CLOSING THIS SESSION:

  1) IMMEDIATE: sidebar -> Data -> Output -> find "results_bundle.zip" -> Download.

  2) DURABLE: click "Save Version" (top right) -> "Save & Run All (Commit)" (NOT Quick
     Save). Wait for it to finish, then open that Version's Output tab and confirm:
       results/pred_hist3_newparser.jsonl
       results/predictions_hist3_newparser.json
       results_bundle.zip
""")


FINAL PERSISTENCE CHECK
[ok]      Hist3 raw predictions (jsonl): 0.17 MB, 1622/1622 rows
[ok]      Hist3 per-row predictions (json, for local paired comparison): 419.5 KB
[checkpoint zip] /kaggle/working/results_bundle.zip (0.07 MB) -- FINAL -- download it now from Data > Output if you want an immediate local copy

full results/ listing: ['confusion_control_nohist_newparser.csv', 'confusion_hist3_newparser.csv', 'confusion_treatment_hist8_newparser.csv', 'metrics_control_nohist_newparser.json', 'metrics_hist3_newparser.json', 'metrics_treatment_hist8_newparser.json', 'paired_comparison.json', 'pred_control_nohist_newparser.jsonl', 'pred_hist3_newparser.jsonl', 'pred_treatment_hist8_newparser.jsonl', 'predictions_control_nohist_newparser.json', 'predictions_hist3_newparser.json', 'predictions_treatment_hist8_newparser.json', 'run_meta_experiment2.json', 'run_meta_experiment3.json', 'smoke_control.jsonl', 'smoke_hist3.jsonl', 'smoke_treatment.jsonl']

ALL CRITICAL FILES PRESENT AND COMPL

## Next step (not part of this notebook)

Upload the resulting `results_bundle.zip`. The three-way paired comparison (NoHist vs Hist8
vs Hist3, same targets/parser/model) will be computed locally against the already-saved
Experiment‑2 `predictions_control_nohist_newparser.json` / `predictions_treatment_hist8_newparser.json` --
no re-running of NoHist or Hist8.
